In [97]:
import base64
import json
from openai import AzureOpenAI 

ENDPOINT = "https://burinakeai-resource.services.ai.azure.com" 
DEPLOYMENT_NAME = "Burinake-vlm"
API_KEY = API_KEY

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=API_KEY,
    api_version="2024-02-15-preview" 
)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

yolo_detection_result = {
    "detected": True,
    "boxes": [
            {
                "x": 0.38,
                "y": 439.95,
                "width": 108.75,
                "height": 118.71,
                "label": "fire",
                "score": 0.3065
            },
            {
                "x": 160.4,
                "y": 366.12,
                "width": 32.95,
                "height": 126.89,
                "label": "fire",
                "score": 0.2962
            },
            {
                "x": 168.09,
                "y": 377.67,
                "width": 23.64,
                "height": 122.48,
                "label": "fire",
                "score": 0.1684
            },
            {
                "x": 157.74,
                "y": 363.89,
                "width": 29.75,
                "height": 103.08,
                "label": "fire",
                "score": 0.1358
            },
            {
                "x": 1.02,
                "y": 4.01,
                "width": 136.59,
                "height": 419.24,
                "label": "smoke",
                "score": 0.113
            }
        ]
}

cctv_metadata = {
    "cctv_id": "CCTV_02", 
    "location": "3층 복도", 
    "captured_at": "2026-07-08T05:15:52"
}

image_sequence_paths = [
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_001.jpg", 
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_002.jpg",
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_003.jpg", 
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_004.jpg", 
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_005.jpg", 
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_006.jpg", 
    r"D:\Study\HumanStudy\burinake\dataset\snapshot3\snapshot_007.jpg",
]

encoded_images = [encode_image(img_path) for img_path in image_sequence_paths]


system_prompt = (
    "You are 'burinake VLM', a highly advanced AI core designed for a smart CCTV-based fire management and emergency dispatch system.\n"
    "You will receive a time-series sequence of sequential frames captured immediately before and after a 1st-stage detector (YOLO) signaled a possible fire or smoke event.\n\n"
    "Your Crucial Missions:\n"
    "1. Strict False Alarm Filtering:\n"
    "   - Standard kitchen activities (steam/smoke from cooking), red glowing neon signs, car taillights, or strong sunsets frequently trigger false alarms in 1st-stage detectors.\n"
    "   - Analyze the temporal context across the image sequence to determine whether this is a genuine, hazardous fire or just a harmless false alarm.\n"
    "2. Comprehensive Fire Inferences:\n"
    "   - If it is verified as an actual fire, assess the current risk level, identify the exact fire source zone or appliance, and deduce the visual cause of the ignition.\n"
    "3. Automated 119 Emergency Report Text (Narrative Style):\n"
    "   - If verified as a true fire, compose a highly accurate, professional emergency report draft in Korean narrative style using the 5W1H rules.\n"
    "   - Heavily incorporate the provided CCTV location and time metadata into this text block so that first responders can pinpoint the site immediately.\n"
    "   - If it is a false alarm, leave this report string completely empty.\n\n"
    "Output Constraint:\n"
    "You must output the final result strictly as a valid JSON object matching the requested schema. No conversational headers, no markdown blocks, no triple backticks (```)."
)

prompt_text = (
    "Please conduct a comprehensive multi-frame analysis on the provided sequential images.\n\n"
    f"[Contextual Data Provided]\n"
    f"- 1st-stage YOLO Detector Raw Output: {json.dumps(yolo_detection_result)}\n"
    f"- System CCTV & Time Metadata: {json.dumps(cctv_metadata)}\n\n"
    "Review the specific regions indicated by YOLO coordinates, analyze the temporal progression, and output the structured data."
)
# ==============================================================


user_content_list = [
    {"type": "text", "text": prompt_text}
]

for b64_str in encoded_images:
    user_content_list.append({
        "type": "image_url",
        "image_url": {
            "url": f"data:image/jpeg;base64,{b64_str}"
        }
    })

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content_list}
]

try:
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        max_completion_tokens=4000,
    )

    if response.choices and response.choices[0].message.content:
        raw_output_text = response.choices[0].message.content
        
        clean_json_text = raw_output_text.strip().strip("```json").strip("```").strip()
        
        parsed_result = json.loads(clean_json_text)
        
        print(json.dumps(parsed_result, ensure_ascii=False, indent=4))
    else:
        print("error")

except Exception as e:
    print(f"{type(e).__name__} - {e}")

{
    "alarm_verification": "fire",
    "confidence": 0.92,
    "cctv_id": "CCTV_02",
    "location": "3층 복도",
    "captured_at": "2026-07-08T05:15:52",
    "yolo_raw": {
        "detected": true,
        "boxes": [
            {
                "x": 0.38,
                "y": 439.95,
                "width": 108.75,
                "height": 118.71,
                "label": "fire",
                "score": 0.3065
            },
            {
                "x": 160.4,
                "y": 366.12,
                "width": 32.95,
                "height": 126.89,
                "label": "fire",
                "score": 0.2962
            },
            {
                "x": 168.09,
                "y": 377.67,
                "width": 23.64,
                "height": 122.48,
                "label": "fire",
                "score": 0.1684
            },
            {
                "x": 157.74,
                "y": 363.89,
                "width": 29.75,
                "height": 10